# AEGIS — Video Anomaly & Threat Detection: Research Prototype (Stage 1)

**Adaptive Environmental Intelligence & Surveillance System**

This notebook builds the core visual-intelligence component only: a temporal model that
takes surveillance video (uploaded file or webcam/RTSP stream) and detects anomalous or
dangerous events — fighting, assault, road accidents, robbery-like activity,
shooting-like activity, explosion, arson, and a general "anomalous vs normal" score.

**Research question:** can an efficient temporal vision system detect dangerous events
in surveillance video, using a lightweight multimodal typed-decision model (Laya Vision)
as a fast semantic verification layer rather than as the primary detector?

This notebook is written for **Google Colab** with a single GPU (T4/L4/A100). It will not
execute end-to-end in a sandbox with no GPU and no dataset access — the code is real and
runnable, but you supply the compute and the data.

---

## Architecture decision (reasoned, not assumed)

### Candidate temporal architectures

| Approach | Temporal modeling | GPU memory | Train time | Inference FPS | Notes |
|---|---|---|---|---|---|
| CNN + LSTM/GRU | Frame-by-frame recurrence over CNN features | Low | Fast | High | Weak at long-range dependencies, sequential (slow to parallelize), but cheap and easy to train on Colab |
| 3D CNN (e.g. R(2+1)D, SlowFast, X3D) | Learns spatiotemporal filters directly from raw clips | High (full 3D convs) | Slow, needs many clips | Medium | Strong accuracy on action recognition benchmarks, but expensive to train from scratch and VRAM-hungry for anything beyond X3D-S/R(2+1)D-18 |
| Temporal Transformer over frozen frame embeddings (TimeSformer/VideoMAE-style, but using a **pretrained frozen 2D backbone + lightweight transformer head**) | Self-attention across sampled frame embeddings | Low–Medium (backbone frozen, only head trains) | Fast (head-only training) | High (frame embeddings can be cached) | Best accuracy/compute tradeoff when full pretraining is out of budget; leverages a strong pretrained image encoder instead of learning visual features from scratch |

### Decision: pretrained 2D visual backbone (frozen/partially fine-tuned) + lightweight temporal Transformer head

Reasoning:
- Training a 3D CNN or a video transformer from scratch needs orders of magnitude more
  compute and data than UCF-Crime/RWF-2000 provide, and Colab GPUs (typically a single
  T4/L4, 16 GB VRAM) can't sustain it in reasonable time.
- A pretrained 2D image backbone (ResNet-50 or a ViT, both are ImageNet-pretrained and
  ship with `torchvision`) already encodes strong spatial features. We don't need to
  relearn "what a person or a car looks like" — we need to learn **how features change
  over time**.
- Freezing (or lightly fine-tuning the last block of) the backbone and training only a
  small Transformer encoder on top of the resulting per-frame embeddings is cheap: frame
  embeddings can be extracted once and cached, so most epochs are just attention over
  8–16-dimensional-reduced embeddings, not repeated CNN forward passes.
- This gives temporal self-attention (better long-range dependency modeling than an
  LSTM/GRU, and fully parallelizable) at a fraction of a 3D CNN's cost.

### Why a temporal model is necessary at all
A fight, an accident, or an explosion is defined by **motion and change across frames**,
not by any single frame's content. A single frame of two people standing close together
is visually indistinguishable from the first frame of a fight. Only the trajectory over
time (proximity change, motion energy, sudden object appearance) disambiguates it.

### Why YOLO alone is insufficient
Object detection answers "what objects are present" (person, car, fire), not "what event
is happening." Two people detected 0.5 m apart could be shaking hands, dancing, or
fighting — object detection cannot tell these apart. YOLO here is a **supporting context
signal** (what objects are involved), not the anomaly detector.

### Why Laya Vision is a verification layer, not the primary detector
Laya Vision answers typed questions about a **single image** in one forward pass — it has
no native temporal memory across frames. It cannot itself tell whether motion occurred
between frame *t-1* and *t*. Used as the primary detector it would essentially be doing
"does this single frame look violent," which throws away exactly the temporal signal
that defines these events. It is used here as a **fast, calibrated, keyframe-level
semantic cross-check**, only invoked on windows the temporal model already flags as
suspicious — because it is small (SmolVLM-256M) and returns calibrated probabilities
rather than free-text, it's cheap enough to run selectively without destroying
throughput.

### Why video-level train/val/test splitting is mandatory
Frames sampled from the same video are highly correlated (same lighting, same actors,
same background). If frames from one video land in both train and val, the model can
memorize background/actor-specific cues instead of learning the event itself — the
validation score becomes an optimistic lie. Every split in this notebook is done at the
video ID level, never the frame or clip level.

### Why temporal smoothing + cooldown
A window-level classifier will always be noisy — a single 2-second window at 0.8
confidence is not evidence of a real event; it's frequently sensor/lighting noise. This
notebook requires **N consecutive windows** above threshold before raising an alert, and
a cooldown period after an alert so a single ongoing event doesn't spam 30 duplicate
alerts.

### Why two outputs (closed-set classification + open anomaly score)
The closed-set classifier can only ever recognize the categories it was trained on
(Fighting, Road Accident, ...). The anomaly score (normal vs. anomalous, trained via the
same labels but collapsed to binary, optionally combined with a reconstruction/novelty
signal) gives a fallback signal for events that don't match any known class — this is
**not** claimed to be a solved open-world detector, it's the minimum foundation a later
AEGIS version needs to eventually handle unseen event types.


## Cell Group 1 — Project setup

In [ ]:
# Install dependencies (Colab). Comment out lines already satisfied by your Colab image.
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip install -q torch torchvision
!pip install -q opencv-python-headless ultralytics scikit-learn pandas matplotlib tqdm seaborn
!pip install -q laya  # Laya / Laya Vision typed-decision model (github.com/r33drichards/laya-vision)


In [ ]:
import os, sys, json, random, time, math, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("WARNING: no GPU detected. Training will fall back to CPU and will be very slow. "
          "In Colab: Runtime > Change runtime type > GPU.")

SEED = 1337
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # False = faster; True if you need bit-exact repro
    torch.backends.cudnn.benchmark = True

set_seed()
print(f"Seed set to {SEED}")


## Cell Group 2 — Configuration

Everything tunable lives in one object. Nothing below hardcodes paths or hyperparameters.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class AegisConfig:
    # --- Paths ---
    dataset_root: str = "/content/datasets"          # expects dataset_root/<dataset_name>/<class>/*.mp4
    ucf_crime_dir: str = "/content/datasets/UCF_Crime"
    rwf2000_dir: str = "/content/datasets/RWF-2000"
    cache_dir: str = "/content/cache"                 # cached preprocessed clip tensors
    output_dir: str = "/content/outputs"
    runs_dir: str = "/content/runs"

    # --- Classes ---
    class_names: List[str] = field(default_factory=lambda: [
        "Normal", "Fighting", "RoadAccident", "Assault", "Shooting",
        "Explosion", "Arson", "Robbery", "Burglary", "Vandalism",
    ])

    # --- Video sampling / clips ---
    image_size: int = 224
    clip_length: int = 16          # frames per temporal window fed to the model
    frame_stride: int = 4          # sample every Nth frame from the source video
    window_stride_sec: float = 1.0 # sliding-window hop for inference (seconds)
    window_length_sec: float = 2.0 # sliding-window length for inference (seconds)

    # --- Backbone / temporal head ---
    backbone_name: str = "resnet50"      # torchvision pretrained backbone
    freeze_backbone: bool = True          # True = extract-and-cache embeddings; False = fine-tune last block
    embed_dim: int = 2048                 # resnet50 pool output dim
    temporal_dim: int = 256               # projected dim fed to the transformer
    temporal_heads: int = 4
    temporal_layers: int = 3
    temporal_ff_dim: int = 512
    dropout: float = 0.2

    # --- Object detector (context features, not the primary detector) ---
    use_yolo: bool = True
    yolo_checkpoint: str = "yolo11n.pt"   # small/fast checkpoint; auto-downloaded by ultralytics
    yolo_sample_every_n_frames: int = 4   # don't run YOLO on every frame

    # --- Laya Vision verification layer ---
    laya_enabled: bool = True
    laya_checkpoint: str = "thaitea/laya-vision-smolvlm-256m"
    laya_prob_threshold: float = 0.55     # only verify windows above this temporal-model probability
    laya_uncertainty_band: tuple = (0.40, 0.60)  # also verify if prediction falls in this "unsure" band
    laya_max_keyframes_per_window: int = 3

    # --- Fusion ---
    fusion_strategy: str = "confidence_weighted"  # "average" | "confidence_weighted" | "threshold_gate"
    laya_fusion_weight: float = 0.4       # base weight for Laya's opinion when it is consulted

    # --- Temporal smoothing / alerting ---
    smoothing_consecutive_windows: int = 3   # must stay above threshold this many windows before alert
    alert_confidence_threshold: float = 0.75
    alert_cooldown_sec: float = 5.0

    # --- Training ---
    batch_size: int = 16
    epochs: int = 25
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    lr_scheduler: str = "cosine"          # "cosine" | "plateau"
    early_stopping_patience: int = 6
    grad_clip_norm: float = 1.0
    use_amp: bool = True                  # mixed precision
    class_balance_strategy: str = "weighted_sampler"  # "weighted_sampler" | "class_weights" | "focal_loss"
    focal_gamma: float = 2.0

    # --- Split ---
    train_frac: float = 0.7
    val_frac: float = 0.15
    test_frac: float = 0.15

    # --- Device ---
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def num_classes(self) -> int:
        return len(self.class_names)


cfg = AegisConfig()
os.makedirs(cfg.cache_dir, exist_ok=True)
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.runs_dir, exist_ok=True)
print(cfg)


## Cell Group 3 — Dependency verification

Fail loudly, not silently, if something required is missing.

In [ ]:
def verify_dependencies(cfg: AegisConfig):
    problems = []

    def check(name, fn):
        try:
            fn()
            print(f"[OK]   {name}")
        except Exception as e:
            problems.append(f"{name}: {e}")
            print(f"[FAIL] {name}: {e}")

    check("torch",        lambda: __import__("torch"))
    check("torchvision",  lambda: __import__("torchvision"))
    check("cv2 (OpenCV)", lambda: __import__("cv2"))
    check("sklearn",      lambda: __import__("sklearn"))
    check("numpy",        lambda: __import__("numpy"))
    check("pandas",       lambda: __import__("pandas"))
    check("matplotlib",   lambda: __import__("matplotlib"))
    check("tqdm",         lambda: __import__("tqdm"))

    if cfg.use_yolo:
        check("ultralytics (YOLO)", lambda: __import__("ultralytics"))

    if cfg.laya_enabled:
        def _laya_check():
            import laya  # noqa: F401
        check("laya", _laya_check)

    if problems:
        raise RuntimeError(
            "Required dependencies are missing or broken:\n  - " + "\n  - ".join(problems) +
            "\nInstall the missing packages before continuing — the pipeline will not silently "
            "degrade for a missing core dependency."
        )
    print("\nAll required dependencies verified.")

verify_dependencies(cfg)


## Cell Group 4 — Dataset preparation

**This notebook does not auto-download UCF-Crime or RWF-2000.** Both require accepting
usage terms from the original authors, and mirroring them from an unofficial source is
exactly the kind of "suspicious mirror" this project should avoid.

Get them from the official sources and place them like this:

```
/content/datasets/
├── UCF_Crime/
│   ├── Fighting/*.mp4
│   ├── RoadAccident/*.mp4
│   ├── Assault/*.mp4
│   ├── Shooting/*.mp4
│   ├── Explosion/*.mp4
│   ├── Arson/*.mp4
│   ├── Robbery/*.mp4
│   ├── Burglary/*.mp4
│   ├── Vandalism/*.mp4
│   └── Normal/*.mp4
└── RWF-2000/
    ├── Fighting/*.avi (or .mp4)
    └── Normal/*.avi
```

- **UCF-Crime**: official page — https://www.crcv.ucf.edu/projects/real-world/ (Sultani et al.,
  CVPR 2018). Released for research use; check the license terms on the page before any
  commercial use.
- **RWF-2000**: official GitHub — https://github.com/mchengny/RWF2000-Video-Database-for-Violence-Detection
  (Cheng et al., ICPR 2020). Access requires filling out the request form linked there —
  it is gated specifically to prevent redistribution, so do not substitute a random mirror.

If you don't have the datasets yet, the cells below will detect that and tell you exactly
what's missing rather than crashing deep into preprocessing.


In [ ]:
def scan_dataset(root: str, class_names):
    """Return {class_name: [video_paths]} for whatever actually exists on disk."""
    found = {}
    missing_classes = []
    if not os.path.isdir(root):
        return found, class_names  # everything missing
    for c in class_names:
        class_dir = os.path.join(root, c)
        if not os.path.isdir(class_dir):
            missing_classes.append(c)
            continue
        vids = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                if f.lower().endswith((".mp4", ".avi", ".mkv", ".mov"))]
        if vids:
            found[c] = vids
        else:
            missing_classes.append(c)
    return found, missing_classes


ucf_found, ucf_missing = scan_dataset(cfg.ucf_crime_dir, cfg.class_names)
total_videos = sum(len(v) for v in ucf_found.values())

if total_videos == 0:
    print("No dataset found yet at", cfg.ucf_crime_dir)
    print("Download UCF-Crime / RWF-2000 per the instructions above, place them at the paths "
          "in `cfg`, then re-run this cell. Every downstream cell depends on this data existing — "
          "there is nothing meaningful to preprocess or train on without it.")
else:
    print(f"Found {total_videos} videos across {len(ucf_found)} classes.")
    if ucf_missing:
        print("Missing/empty classes (will be skipped):", ucf_missing)
    for c, vids in ucf_found.items():
        print(f"  {c:15s} {len(vids):4d} videos")


## Cell Group 5 — Dataset exploration

In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt

def probe_video(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = n_frames / fps if fps > 0 else 0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    return dict(path=path, fps=fps, n_frames=n_frames, duration_sec=duration, width=w, height=h)


def explore_dataset(found: dict, sample_per_class: int = 15):
    rows = []
    for cls, vids in found.items():
        for p in vids[:sample_per_class]:
            try:
                info = probe_video(p)
                info["class"] = cls
                rows.append(info)
            except Exception as e:
                print(f"  could not probe {p}: {e}")
    df = pd.DataFrame(rows)
    if df.empty:
        print("Nothing to explore yet — dataset is empty.")
        return df

    print(df.groupby("class")["duration_sec"].describe()[["count", "mean", "min", "max"]])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["class"].value_counts().plot(kind="bar", ax=axes[0], title="Sampled videos per class")
    df["duration_sec"].hist(bins=30, ax=axes[1])
    axes[1].set_title("Duration distribution (sec)")
    plt.tight_layout()
    plt.show()
    return df


if total_videos > 0:
    explore_df = explore_dataset(ucf_found)


## Cell Group 6 — Video preprocessing

Frame sampling → resize → normalize → temporal clip tensor, with disk caching so the
same video is never decoded twice across epochs.


In [ ]:
import hashlib
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess_tf = T.Compose([
    T.ToPILImage(),
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def _cache_key(video_path: str, start_frame: int, clip_length: int, stride: int) -> str:
    raw = f"{video_path}|{start_frame}|{clip_length}|{stride}|{cfg.image_size}"
    return hashlib.md5(raw.encode()).hexdigest()


def sample_clip(video_path: str, start_frame: int, clip_length: int = None, stride: int = None):
    """Read `clip_length` frames starting at `start_frame`, every `stride`-th frame.
    Returns a (clip_length, 3, H, W) float tensor, or None if the video is too short/corrupt.
    Cached to disk under cfg.cache_dir."""
    clip_length = clip_length or cfg.clip_length
    stride = stride or cfg.frame_stride

    key = _cache_key(video_path, start_frame, clip_length, stride)
    cache_path = os.path.join(cfg.cache_dir, key + ".pt")
    if os.path.exists(cache_path):
        try:
            return torch.load(cache_path)
        except Exception:
            pass  # fall through and re-decode if cache is corrupt

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    frames = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    for i in range(clip_length):
        for _ in range(stride):
            ok, frame = cap.read()
            if not ok:
                break
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(preprocess_tf(frame))
    cap.release()

    if len(frames) < clip_length:
        return None  # too short — caller should skip this window

    clip = torch.stack(frames, dim=0)  # (T, C, H, W)
    try:
        torch.save(clip, cache_path)
    except Exception:
        pass
    return clip


def video_metadata(video_path: str):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return fps, n_frames


print("Preprocessing pipeline ready. Cache dir:", cfg.cache_dir)


## Cell Group 7 — Dataset class and video-level split

Split happens on **video ID**, before any clip is generated, so no clip from a
validation/test video can leak into training.


In [ ]:
from torch.utils.data import Dataset

def video_level_split(found: dict, cfg: AegisConfig, seed: int = SEED):
    """Split at the video level. Returns three dicts: {class: [paths]} for train/val/test."""
    rng = random.Random(seed)
    train, val, test = {}, {}, {}
    for cls, vids in found.items():
        vids = vids[:]
        rng.shuffle(vids)
        n = len(vids)
        n_train = max(1, int(n * cfg.train_frac))
        n_val = max(1, int(n * cfg.val_frac)) if n > 2 else 0
        train[cls] = vids[:n_train]
        val[cls] = vids[n_train:n_train + n_val]
        test[cls] = vids[n_train + n_val:]
        if not test[cls]:  # guarantee non-empty test where possible
            if len(train[cls]) > 1:
                test[cls] = [train[cls].pop()]
    return train, val, test


class VideoClipDataset(Dataset):
    """Each item is one temporal clip (clip_length frames), its class label,
    the anomaly label (binary), the source video id, and the clip's start timestamp."""

    def __init__(self, split_dict: dict, cfg: AegisConfig, clips_per_video: int = 4, training: bool = True):
        self.cfg = cfg
        self.training = training
        self.class_to_idx = {c: i for i, c in enumerate(cfg.class_names)}
        self.samples = []  # (video_path, start_frame, class_idx)

        for cls, vids in split_dict.items():
            if cls not in self.class_to_idx:
                continue
            cidx = self.class_to_idx[cls]
            for vp in vids:
                fps, n_frames = video_metadata(vp)
                needed = cfg.clip_length * cfg.frame_stride
                if n_frames <= needed:
                    self.samples.append((vp, 0, cidx))
                    continue
                max_start = n_frames - needed
                starts = sorted(random.Random(hash(vp) % (2**31)).sample(
                    range(0, max_start), min(clips_per_video, max_start)))
                for s in starts:
                    self.samples.append((vp, s, cidx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, start_frame, class_idx = self.samples[idx]
        clip = sample_clip(video_path, start_frame)
        if clip is None:
            # fall back to a different sample rather than crashing the loader
            return self.__getitem__((idx + 1) % len(self.samples))
        anomaly_label = 0 if self.cfg.class_names[class_idx] == "Normal" else 1
        return {
            "clip": clip,                       # (T, C, H, W)
            "class_label": class_idx,
            "anomaly_label": anomaly_label,
            "video_path": video_path,
            "start_frame": start_frame,
        }


if total_videos > 0:
    train_split, val_split, test_split = video_level_split(ucf_found, cfg)
    print("Train videos:", sum(len(v) for v in train_split.values()))
    print("Val videos:  ", sum(len(v) for v in val_split.values()))
    print("Test videos: ", sum(len(v) for v in test_split.values()))

    # sanity check: no video path should appear in more than one split
    train_paths = {p for v in train_split.values() for p in v}
    val_paths = {p for v in val_split.values() for p in v}
    test_paths = {p for v in test_split.values() for p in v}
    assert not (train_paths & val_paths), "Data leakage: video in both train and val"
    assert not (train_paths & test_paths), "Data leakage: video in both train and test"
    assert not (val_paths & test_paths), "Data leakage: video in both val and test"
    print("No video-level leakage between splits — confirmed.")


## Cell Group 8 — Baseline model (Baseline A: temporal-pooling only)

Simplest credible baseline: pretrained backbone → mean-pool frame embeddings over time →
linear classifier. No attention, no temporal ordering awareness. This exists specifically
so the improved model (Cell Group 9) has something concrete to beat.


In [ ]:
import torch.nn as nn
import torchvision.models as tvm

def build_backbone(name: str, freeze: bool = True):
    if name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
    elif name == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
    else:
        raise ValueError(f"Unsupported backbone: {name}")

    if freeze:
        for p in m.parameters():
            p.requires_grad = False
    else:
        # fine-tune only the last residual block, keep everything else frozen
        for p in m.parameters():
            p.requires_grad = False
        for p in m.layer4.parameters():
            p.requires_grad = True
    return m, feat_dim


class BaselinePoolingModel(nn.Module):
    """Baseline A: backbone -> mean-pool over time -> classifier heads."""

    def __init__(self, cfg: AegisConfig):
        super().__init__()
        self.backbone, feat_dim = build_backbone(cfg.backbone_name, cfg.freeze_backbone)
        self.event_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(cfg.dropout),
            nn.Linear(256, cfg.num_classes),
        )
        self.anomaly_head = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(), nn.Dropout(cfg.dropout),
            nn.Linear(128, 1),
        )

    def forward(self, clip):  # clip: (B, T, C, H, W)
        B, T, C, H, W = clip.shape
        x = clip.view(B * T, C, H, W)
        feats = self.backbone(x).view(B, T, -1)   # (B, T, feat_dim)
        pooled = feats.mean(dim=1)                # (B, feat_dim)
        event_logits = self.event_head(pooled)
        anomaly_logit = self.anomaly_head(pooled).squeeze(-1)
        return event_logits, anomaly_logit


print("BaselinePoolingModel defined.")


## Cell Group 9 — Improved temporal model (AEGIS primary detector)

Backbone → per-frame embedding projection → learned positional encoding → Transformer
encoder over the time dimension → pooled representation → dual heads (event
classification, anomaly score). This is the architecture selected in the reasoning
section above: attention gives it long-range temporal context an LSTM/GRU struggles
with, at a fraction of a 3D CNN's training cost, because the backbone stays frozen (or
only its last block trains).


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, dim)

    def forward(self, x):  # x: (B, T, dim)
        return x + self.pe[:, :x.size(1), :]


class AegisTemporalModel(nn.Module):
    """Primary AEGIS event/anomaly detector."""

    def __init__(self, cfg: AegisConfig):
        super().__init__()
        self.backbone, feat_dim = build_backbone(cfg.backbone_name, cfg.freeze_backbone)
        self.project = nn.Linear(feat_dim, cfg.temporal_dim)
        self.pos_enc = PositionalEncoding(cfg.temporal_dim, max_len=cfg.clip_length + 4)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.temporal_dim,
            nhead=cfg.temporal_heads,
            dim_feedforward=cfg.temporal_ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            activation="gelu",
        )
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg.temporal_layers)

        # learned CLS-style token that attends to the whole clip and is used for pooling
        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.temporal_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.event_head = nn.Sequential(
            nn.LayerNorm(cfg.temporal_dim),
            nn.Linear(cfg.temporal_dim, 256), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(256, cfg.num_classes),
        )
        self.anomaly_head = nn.Sequential(
            nn.LayerNorm(cfg.temporal_dim),
            nn.Linear(cfg.temporal_dim, 128), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(128, 1),
        )

    def forward(self, clip, object_features=None):
        """clip: (B, T, C, H, W). object_features: optional (B, obj_dim) to concat post-hoc
        by the fusion model in Cell Group 12 — this model stays clip-only by design so it can
        be evaluated standalone as Baseline A/C in the ablation table."""
        B, T, C, H, W = clip.shape
        x = clip.view(B * T, C, H, W)
        with torch.set_grad_enabled(any(p.requires_grad for p in self.backbone.parameters())):
            feats = self.backbone(x).view(B, T, -1)
        tokens = self.project(feats)                      # (B, T, temporal_dim)
        tokens = self.pos_enc(tokens)

        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)           # (B, T+1, temporal_dim)

        encoded = self.temporal_encoder(tokens)
        pooled = encoded[:, 0, :]                           # take the CLS token's output

        event_logits = self.event_head(pooled)
        anomaly_logit = self.anomaly_head(pooled).squeeze(-1)
        return event_logits, anomaly_logit, pooled


print("AegisTemporalModel defined "
      f"(backbone={cfg.backbone_name}, temporal_dim={cfg.temporal_dim}, "
      f"layers={cfg.temporal_layers}, heads={cfg.temporal_heads}).")


## Cell Group 10 — Object feature extraction (YOLO, supporting signal only)

YOLO is run on a sparse subset of frames (`cfg.yolo_sample_every_n_frames`), not every
frame — it exists to give the fusion model contextual counts (people present, vehicles
present, fire detected), never to decide the event itself.


In [ ]:
def load_yolo(cfg: AegisConfig):
    if not cfg.use_yolo:
        return None
    try:
        from ultralytics import YOLO
        return YOLO(cfg.yolo_checkpoint)
    except Exception as e:
        print(f"YOLO unavailable ({e}) — continuing without object features. "
              f"The temporal model and Laya layer still work; only the object-context "
              f"channel in the fusion model is disabled.")
        return None


yolo_model = load_yolo(cfg)

# Object classes we actually care about for surveillance context
RELEVANT_YOLO_CLASSES = {"person", "car", "truck", "motorcycle", "bicycle", "bus"}

def extract_object_features(clip_frames_rgb: list, yolo_model, every_n: int = None):
    """clip_frames_rgb: list of HxWx3 uint8 RGB numpy arrays (raw, pre-normalization).
    Returns a fixed-size feature vector: counts of each relevant class, averaged over
    the sampled frames."""
    every_n = every_n or cfg.yolo_sample_every_n_frames
    counts = {c: [] for c in RELEVANT_YOLO_CLASSES}

    if yolo_model is None:
        return np.zeros(len(RELEVANT_YOLO_CLASSES), dtype=np.float32)

    for i, frame in enumerate(clip_frames_rgb):
        if i % every_n != 0:
            continue
        result = yolo_model.predict(frame, verbose=False)[0]
        frame_counts = {c: 0 for c in RELEVANT_YOLO_CLASSES}
        for cls_id in result.boxes.cls.tolist():
            name = yolo_model.names[int(cls_id)]
            if name in RELEVANT_YOLO_CLASSES:
                frame_counts[name] += 1
        for c in RELEVANT_YOLO_CLASSES:
            counts[c].append(frame_counts[c])

    vec = np.array([np.mean(v) if v else 0.0 for v in counts.values()], dtype=np.float32)
    return vec

OBJECT_FEATURE_DIM = len(RELEVANT_YOLO_CLASSES)
print(f"Object feature dim: {OBJECT_FEATURE_DIM} ({sorted(RELEVANT_YOLO_CLASSES)})")


## Cell Group 11 — Laya Vision verification layer

Uses the real `laya-vision` API (`laya.load_vlm(...)`, `agent.predict(state, questions)`),
called **only** on keyframes from windows the temporal model already flagged as
suspicious (`laya_prob_threshold`) or genuinely uncertain about (`laya_uncertainty_band`).
Typed `noul`/`score` questions are used instead of free-form generation — cheaper,
calibrated, and impossible to hallucinate a malformed answer from.


In [ ]:
_laya_agent = None

def get_laya_agent(cfg: AegisConfig):
    global _laya_agent
    if not cfg.laya_enabled:
        return None
    if _laya_agent is not None:
        return _laya_agent
    try:
        import laya
        _laya_agent = laya.load_vlm(cfg.laya_checkpoint)
        print(f"Laya Vision loaded: {cfg.laya_checkpoint}")
    except Exception as e:
        print(f"Laya Vision unavailable ({e}) — the temporal model still runs standalone; "
              f"only the verification/fusion step is skipped.")
        _laya_agent = None
    return _laya_agent


# Typed questions per candidate event class. Kept short and specific — Laya scores each
# in a single forward pass per image, so precision in the question matters more than quantity.
LAYA_QUESTIONS = {
    "Fighting": {"type": "noul", "instructions": "Does this frame show people in physical confrontation, such as hitting, grappling, or striking each other?"},
    "Assault":  {"type": "noul", "instructions": "Does this frame show one person physically attacking another?"},
    "RoadAccident": {"type": "noul", "instructions": "Does this frame show a vehicle collision or a vehicle that has crashed?"},
    "Shooting": {"type": "noul", "instructions": "Does this frame show a person holding or firing a firearm?"},
    "Explosion": {"type": "noul", "instructions": "Does this frame show an explosion, fireball, or blast?"},
    "Arson": {"type": "noul", "instructions": "Does this frame show fire deliberately burning a structure or object?"},
    "Robbery": {"type": "noul", "instructions": "Does this frame show one person forcibly taking property from another, e.g. at gunpoint or by force?"},
    "severity": {"type": "score", "instructions": "How severe is the apparent danger or violence in this frame?",
                 "criteria": ["none: no danger or violence visible", "mild: tense but no physical harm",
                              "moderate: physical altercation or clear hazard", "severe: serious violence, injury, or major hazard"]},
}


def select_keyframes(clip_frames_rgb: list, k: int) -> list:
    """Evenly-spaced keyframes from a clip — cheap, deterministic, avoids biasing toward
    the first or last frame of the window."""
    n = len(clip_frames_rgb)
    if n <= k:
        return clip_frames_rgb
    idxs = np.linspace(0, n - 1, k).astype(int)
    return [clip_frames_rgb[i] for i in idxs]


def laya_verify_clip(clip_frames_rgb: list, candidate_classes: list, cfg: AegisConfig):
    """Runs Laya Vision on a few keyframes of a suspicious window, only for the classes
    the temporal model actually flagged as plausible. Returns a dict:
    {class_name: {"laya_prob": float, "severity_score": float}}"""
    agent = get_laya_agent(cfg)
    if agent is None:
        return {}

    from PIL import Image
    keyframes = select_keyframes(clip_frames_rgb, cfg.laya_max_keyframes_per_window)

    per_class_scores = {c: [] for c in candidate_classes if c in LAYA_QUESTIONS}
    severity_scores = []

    for frame in keyframes:
        image = Image.fromarray(frame)
        questions = {c: LAYA_QUESTIONS[c] for c in per_class_scores}
        questions["severity"] = LAYA_QUESTIONS["severity"]

        result = agent.predict({"image": image}, questions)

        for c in per_class_scores:
            per_class_scores[c].append(result["answers"][c]["noul"])
        severity_scores.append(result["answers"]["severity"]["score"])

    out = {}
    for c, scores in per_class_scores.items():
        out[c] = {
            "laya_prob": float(np.mean(scores)) if scores else 0.0,
            "severity_score": float(np.mean(severity_scores)) if severity_scores else 0.0,
        }
    return out


def should_verify_with_laya(top_prob: float, cfg: AegisConfig) -> bool:
    """Cascade gate: only call Laya when the temporal model is confident-but-not-certain,
    or genuinely unsure — never on every window (would destroy throughput)."""
    if not cfg.laya_enabled:
        return False
    lo, hi = cfg.laya_uncertainty_band
    is_suspicious = top_prob >= cfg.laya_prob_threshold
    is_uncertain = lo <= top_prob <= hi
    return is_suspicious or is_uncertain


print("Laya Vision verification layer defined. Call get_laya_agent(cfg) once to warm-load it.")


## Cell Group 12 — Decision fusion

Not a blind average. `confidence_weighted`: Laya's opinion is weighted by how confident
*it* is (its own calibrated probability distance from 0.5), so a wishy-washy Laya answer
doesn't drag down a confident temporal-model prediction, and vice versa.


In [ ]:
def fuse_predictions(temporal_prob: float, laya_result: dict, event_class: str, cfg: AegisConfig):
    """temporal_prob: the temporal model's probability for `event_class`.
    laya_result: output of laya_verify_clip for this window (may be empty if not consulted).
    Returns (final_prob, contributors_dict) for transparency/logging."""
    if not laya_result or event_class not in laya_result:
        return temporal_prob, {"temporal": temporal_prob, "laya": None, "strategy": "temporal_only"}

    laya_prob = laya_result[event_class]["laya_prob"]

    if cfg.fusion_strategy == "average":
        final = 0.5 * temporal_prob + 0.5 * laya_prob

    elif cfg.fusion_strategy == "threshold_gate":
        # Laya can veto: if it strongly disagrees (says "no" with high confidence),
        # damp the temporal model's score rather than trusting it blindly.
        if laya_prob < 0.15:
            final = temporal_prob * 0.3
        else:
            final = max(temporal_prob, laya_prob * 0.9)

    else:  # confidence_weighted (default)
        laya_confidence = abs(laya_prob - 0.5) * 2       # 0 (unsure) .. 1 (certain)
        w = cfg.laya_fusion_weight * laya_confidence
        final = (1 - w) * temporal_prob + w * laya_prob

    final = float(np.clip(final, 0.0, 1.0))
    return final, {"temporal": temporal_prob, "laya": laya_prob, "strategy": cfg.fusion_strategy}


print(f"Fusion strategy: {cfg.fusion_strategy}")


## Cell Group 13 — Training

Handles class imbalance (`weighted_sampler` / `class_weights` / `focal_loss` — configurable),
mixed precision, gradient clipping, cosine or plateau LR scheduling, early stopping, and
saves the best checkpoint by validation macro-F1.


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def compute_class_weights(split_dict: dict, class_names: list):
    counts = np.array([len(split_dict.get(c, [])) for c in class_names], dtype=np.float32)
    counts = np.clip(counts, 1, None)
    weights = counts.sum() / (len(class_names) * counts)
    return torch.tensor(weights, dtype=torch.float32)


def build_dataloader(dataset: VideoClipDataset, cfg: AegisConfig, train_split: dict = None, shuffle=True):
    if cfg.class_balance_strategy == "weighted_sampler" and train_split is not None:
        class_counts = np.array([len(train_split.get(c, [])) for c in cfg.class_names], dtype=np.float32)
        class_counts = np.clip(class_counts, 1, None)
        class_weight_lookup = 1.0 / class_counts
        sample_weights = [class_weight_lookup[lbl] for _, _, lbl in dataset.samples]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        return DataLoader(dataset, batch_size=cfg.batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    return DataLoader(dataset, batch_size=cfg.batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)


def train_aegis_model(model, train_loader, val_loader, cfg: AegisConfig, class_weights=None, run_name="aegis"):
    device = cfg.device
    model = model.to(device)

    if cfg.class_balance_strategy == "focal_loss":
        event_criterion = FocalLoss(gamma=cfg.focal_gamma, weight=class_weights.to(device) if class_weights is not None else None)
    else:
        w = class_weights.to(device) if (cfg.class_balance_strategy == "class_weights" and class_weights is not None) else None
        event_criterion = nn.CrossEntropyLoss(weight=w)
    anomaly_criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.learning_rate, weight_decay=cfg.weight_decay,
    )
    if cfg.lr_scheduler == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)

    scaler = GradScaler(enabled=cfg.use_amp)

    best_val_f1 = -1.0
    epochs_without_improvement = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}

    run_dir = os.path.join(cfg.runs_dir, f"{run_name}_{time.strftime('%Y%m%d_%H%M%S')}")
    os.makedirs(run_dir, exist_ok=True)

    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            clip = batch["clip"].to(device, non_blocking=True)
            event_labels = batch["class_label"].to(device)
            anomaly_labels = batch["anomaly_label"].float().to(device)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=cfg.use_amp):
                event_logits, anomaly_logit, _ = model(clip)
                loss = event_criterion(event_logits, event_labels) + anomaly_criterion(anomaly_logit, anomaly_labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * clip.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        val_loss, val_f1, _, _ = evaluate_model(model, val_loader, cfg, event_criterion, anomaly_criterion)

        if cfg.lr_scheduler == "cosine":
            scheduler.step()
        else:
            scheduler.step(val_f1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        print(f"Epoch {epoch+1:02d}/{cfg.epochs} | train_loss={train_loss:.4f} "
              f"| val_loss={val_loss:.4f} | val_macro_F1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_without_improvement = 0
            save_checkpoint(model, optimizer, epoch, cfg, val_f1, history,
                             path=os.path.join(cfg.output_dir, "aegis_best.pt"))
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= cfg.early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1} (no val_F1 improvement for "
                      f"{cfg.early_stopping_patience} epochs).")
                break

    save_checkpoint(model, optimizer, epoch, cfg, best_val_f1, history,
                     path=os.path.join(cfg.output_dir, "aegis_final.pt"))

    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg.__dict__, f, indent=2, default=str)

    return history, best_val_f1


### Checkpoint save / load — real `.pt` files, not renamed placeholders

In [ ]:
def save_checkpoint(model, optimizer, epoch, cfg: AegisConfig, val_f1, history, path):
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "class_names": cfg.class_names,
        "config": {k: v for k, v in cfg.__dict__.items()},
        "metrics": {"val_macro_f1": val_f1, "history": history},
        "model_architecture": model.__class__.__name__,
    }, path)
    print(f"Saved checkpoint: {path} (val_macro_F1={val_f1:.4f})")


def load_checkpoint(path, cfg: AegisConfig, model_cls=None):
    """Loads a checkpoint from scratch and reconstructs the model — proves the .pt
    file is actually sufficient to reproduce inference, not just a state_dict fragment."""
    ckpt = torch.load(path, map_location=cfg.device)
    model_cls = model_cls or AegisTemporalModel
    model = model_cls(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(cfg.device).eval()
    print(f"Loaded {ckpt['model_architecture']} from {path} "
          f"(epoch {ckpt['epoch']}, val_macro_F1={ckpt['metrics']['val_macro_f1']:.4f})")
    return model, ckpt


## Evaluation & metrics

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_curve, auc as sk_auc,
)
import seaborn as sns

@torch.no_grad()
def evaluate_model(model, loader, cfg: AegisConfig, event_criterion=None, anomaly_criterion=None):
    model.eval()
    device = cfg.device
    all_preds, all_labels, all_anomaly_probs, all_anomaly_labels = [], [], [], []
    total_loss = 0.0

    for batch in loader:
        clip = batch["clip"].to(device)
        event_labels = batch["class_label"].to(device)
        anomaly_labels = batch["anomaly_label"].float().to(device)

        event_logits, anomaly_logit, _ = model(clip)

        if event_criterion is not None and anomaly_criterion is not None:
            loss = event_criterion(event_logits, event_labels) + anomaly_criterion(anomaly_logit, anomaly_labels)
            total_loss += loss.item() * clip.size(0)

        preds = event_logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(event_labels.cpu().numpy())
        all_anomaly_probs.extend(torch.sigmoid(anomaly_logit).cpu().numpy())
        all_anomaly_labels.extend(anomaly_labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset) if (event_criterion is not None) else 0.0
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, macro_f1, (all_labels, all_preds), (all_anomaly_labels, all_anomaly_probs)


def full_evaluation_report(model, loader, cfg: AegisConfig, split_name="test"):
    _, macro_f1, (labels, preds), (anomaly_labels, anomaly_probs) = evaluate_model(model, loader, cfg)

    report = classification_report(labels, preds, target_names=cfg.class_names,
                                    zero_division=0, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(os.path.join(cfg.output_dir, f"classification_report_{split_name}.csv"))
    print(report_df)

    cm = confusion_matrix(labels, preds, labels=list(range(cfg.num_classes)))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=cfg.class_names, yticklabels=cfg.class_names, cmap="Blues")
    plt.title(f"Confusion matrix — {split_name}")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.output_dir, "confusion_matrix.png"))
    plt.show()

    metrics = {"macro_f1": macro_f1}
    try:
        metrics["anomaly_roc_auc"] = roc_auc_score(anomaly_labels, anomaly_probs)
        prec, rec, _ = precision_recall_curve(anomaly_labels, anomaly_probs)
        metrics["anomaly_pr_auc"] = sk_auc(rec, prec)
    except ValueError:
        metrics["anomaly_roc_auc"] = None
        metrics["anomaly_pr_auc"] = None

    with open(os.path.join(cfg.output_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print("\nSummary metrics:", metrics)
    return metrics


def benchmark_inference_speed(model, cfg: AegisConfig, n_iters=30):
    model.eval()
    dummy = torch.randn(1, cfg.clip_length, 3, cfg.image_size, cfg.image_size, device=cfg.device)
    # warmup
    with torch.no_grad():
        for _ in range(5):
            model(dummy)
    torch.cuda.synchronize() if cfg.device == "cuda" else None
    start = time.time()
    with torch.no_grad():
        for _ in range(n_iters):
            model(dummy)
    torch.cuda.synchronize() if cfg.device == "cuda" else None
    elapsed = time.time() - start
    latency_ms = (elapsed / n_iters) * 1000
    fps = cfg.clip_length / (latency_ms / 1000)
    print(f"Latency: {latency_ms:.1f} ms/window | Effective FPS: {fps:.1f} "
          f"(clip_length={cfg.clip_length} frames/window)")
    return latency_ms, fps


## Run training

This is the cell that actually trains the AEGIS temporal model, once the dataset is in
place. Skip straight to "Inference API" below if you're loading an already-trained
`aegis_best.pt` instead.


In [ ]:
if total_videos > 0:
    train_ds = VideoClipDataset(train_split, cfg, clips_per_video=4, training=True)
    val_ds   = VideoClipDataset(val_split,   cfg, clips_per_video=2, training=False)
    test_ds  = VideoClipDataset(test_split,  cfg, clips_per_video=2, training=False)

    print(f"Train clips: {len(train_ds)} | Val clips: {len(val_ds)} | Test clips: {len(test_ds)}")

    train_loader = build_dataloader(train_ds, cfg, train_split=train_split, shuffle=True)
    val_loader   = build_dataloader(val_ds, cfg, shuffle=False)
    test_loader  = build_dataloader(test_ds, cfg, shuffle=False)

    class_weights = compute_class_weights(train_split, cfg.class_names)

    aegis_model = AegisTemporalModel(cfg)
    history, best_val_f1 = train_aegis_model(
        aegis_model, train_loader, val_loader, cfg,
        class_weights=class_weights, run_name="aegis_temporal",
    )

    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.plot(history["val_f1"], label="val_macro_f1")
    plt.legend(); plt.title("Training curves"); plt.xlabel("epoch")
    plt.savefig(os.path.join(cfg.output_dir, "training_curves.png"))
    plt.show()

    best_model, best_ckpt = load_checkpoint(os.path.join(cfg.output_dir, "aegis_best.pt"), cfg)
    test_metrics = full_evaluation_report(best_model, test_loader, cfg, split_name="test")
    benchmark_inference_speed(best_model, cfg)
else:
    print("Skipping training — no dataset found. Populate cfg.ucf_crime_dir and re-run from "
          "Cell Group 4 onward.")


## Inference API — `predict_video()`

Sliding-window inference over an uploaded video: run the temporal model on overlapping
windows, cascade into Laya where the gate says so, fuse, smooth over consecutive
windows, and only raise an alert once confidence is sustained (with a cooldown so one
event doesn't spam 30 alerts).


In [ ]:
class AlertStateMachine:
    """Implements smoothing_consecutive_windows + alert_cooldown_sec from cfg."""

    def __init__(self, cfg: AegisConfig):
        self.cfg = cfg
        self.consecutive_count = 0
        self.last_alert_time = -1e9
        self.active_event = None

    def update(self, timestamp: float, event_class: str, confidence: float):
        cfg = self.cfg
        above_threshold = confidence >= cfg.alert_confidence_threshold

        if above_threshold and event_class == self.active_event:
            self.consecutive_count += 1
        elif above_threshold:
            self.active_event = event_class
            self.consecutive_count = 1
        else:
            self.active_event = None
            self.consecutive_count = 0

        should_fire = (
            above_threshold
            and self.consecutive_count >= cfg.smoothing_consecutive_windows
            and (timestamp - self.last_alert_time) >= cfg.alert_cooldown_sec
        )
        if should_fire:
            self.last_alert_time = timestamp
        return should_fire


@torch.no_grad()
def predict_video(video_path: str, model, cfg: AegisConfig, yolo_model=None, save_annotated=True):
    model.eval()
    fps, n_frames = video_metadata(video_path)
    window_frames = int(cfg.window_length_sec * fps)
    hop_frames = int(cfg.window_stride_sec * fps)

    events, alerts = [], []
    alert_machine = AlertStateMachine(cfg)
    start_time = time.time()

    cap = cv2.VideoCapture(video_path)
    writer = None
    if save_annotated:
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_path = os.path.join(cfg.output_dir, "sample_annotated_video.mp4")
        writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    frame_start = 0
    overall_anomaly_scores = []

    while frame_start + window_frames <= n_frames:
        clip = sample_clip(video_path, frame_start, clip_length=cfg.clip_length,
                            stride=max(1, window_frames // cfg.clip_length))
        if clip is None:
            frame_start += hop_frames
            continue

        clip_batch = clip.unsqueeze(0).to(cfg.device)
        event_logits, anomaly_logit, _ = model(clip_batch)
        event_probs = torch.softmax(event_logits, dim=1)[0].cpu().numpy()
        anomaly_prob = torch.sigmoid(anomaly_logit)[0].item()
        overall_anomaly_scores.append(anomaly_prob)

        top_idx = int(np.argmax(event_probs))
        top_class = cfg.class_names[top_idx]
        top_prob = float(event_probs[top_idx])

        laya_result = {}
        if top_class != "Normal" and should_verify_with_laya(top_prob, cfg):
            raw_frames = [((f.numpy().transpose(1, 2, 0) *
                            np.array([0.229, 0.224, 0.225]) +
                            np.array([0.485, 0.456, 0.406])) * 255).clip(0, 255).astype(np.uint8)
                          for f in clip]
            candidate_classes = [c for c, p in zip(cfg.class_names, event_probs) if p > 0.1 and c != "Normal"]
            laya_result = laya_verify_clip(raw_frames, candidate_classes, cfg)

        final_prob, contributors = fuse_predictions(top_prob, laya_result, top_class, cfg)

        window_start_sec = frame_start / fps
        window_end_sec = (frame_start + window_frames) / fps

        fired = alert_machine.update(window_end_sec, top_class, final_prob)
        event_record = {
            "event": top_class,
            "start_time": round(window_start_sec, 2),
            "end_time": round(window_end_sec, 2),
            "confidence": round(final_prob, 3),
            "anomaly_score": round(anomaly_prob, 3),
            "verification": {"laya": laya_result.get(top_class, {}).get("laya_prob")} if laya_result else None,
        }
        events.append(event_record)
        if fired:
            alerts.append(event_record)

        frame_start += hop_frames

    cap.release()

    if writer is not None:
        _annotate_and_write_video(video_path, cfg, events, writer, fps)
        writer.release()

    elapsed = time.time() - start_time
    result = {
        "events": events,
        "overall_anomaly_score": float(np.mean(overall_anomaly_scores)) if overall_anomaly_scores else 0.0,
        "processing_time": round(elapsed, 2),
        "fps": round(n_frames / elapsed, 1) if elapsed > 0 else None,
        "alerts": alerts,
    }

    with open(os.path.join(cfg.output_dir, "inference_results.json"), "w") as f:
        json.dump(result, f, indent=2)

    return result


def _annotate_and_write_video(video_path, cfg, events, writer, fps):
    """Draws a lightweight AEGIS overlay onto each frame using the nearest window's result."""
    cap = cv2.VideoCapture(video_path)
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = frame_idx / fps
        current = next((e for e in events if e["start_time"] <= t <= e["end_time"]), None)

        if current and current["event"] != "Normal" and current["confidence"] >= cfg.alert_confidence_threshold:
            status, color = "ALERT", (0, 0, 255)
        elif current:
            status, color = "NORMAL", (0, 200, 0)
        else:
            status, color, current = "NORMAL", (0, 200, 0), {"event": "Normal", "confidence": 0.0, "anomaly_score": 0.0}

        cv2.rectangle(frame, (0, 0), (330, 100), (20, 20, 20), -1)
        cv2.putText(frame, "AEGIS", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f"EVENT: {current['event']}", (10, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        cv2.putText(frame, f"CONF: {current['confidence']*100:.0f}%", (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        cv2.putText(frame, f"ANOMALY: {current['anomaly_score']*100:.0f}%", (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        cv2.putText(frame, f"STATUS: {status}", (10, 96), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        writer.write(frame)
        frame_idx += 1
    cap.release()

print("predict_video() ready.")


## Live video mode — `predict_live()`

Same pipeline, applied to a rolling frame buffer instead of a file. Runs in a plain
Python loop (Colab can't render a live OpenCV window directly — this prints
alerts/status to stdout; swap the print for a real display when running locally).


In [ ]:
def predict_live(model, cfg: AegisConfig, source=0, max_seconds=None):
    """source=0 for default webcam, or an RTSP URL string. Runs until interrupted or
    max_seconds elapses."""
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video source: {source}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    window_frames = int(cfg.window_length_sec * fps)
    buffer = []
    alert_machine = AlertStateMachine(cfg)
    t0 = time.time()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Stream ended or camera disconnected.")
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            buffer.append(preprocess_tf(rgb))
            if len(buffer) > window_frames:
                buffer.pop(0)

            if len(buffer) == window_frames:
                idxs = np.linspace(0, window_frames - 1, cfg.clip_length).astype(int)
                clip = torch.stack([buffer[i] for i in idxs]).unsqueeze(0).to(cfg.device)

                with torch.no_grad():
                    event_logits, anomaly_logit, _ = model(clip)
                event_probs = torch.softmax(event_logits, dim=1)[0].cpu().numpy()
                anomaly_prob = torch.sigmoid(anomaly_logit)[0].item()
                top_idx = int(np.argmax(event_probs))
                top_class, top_prob = cfg.class_names[top_idx], float(event_probs[top_idx])

                elapsed = time.time() - t0
                fired = alert_machine.update(elapsed, top_class, top_prob)
                status = "ALERT" if fired else ("NORMAL" if top_class == "Normal" else "watching")
                print(f"EVENT: {top_class:12s} CONFIDENCE: {top_prob*100:5.1f}%  "
                      f"STATUS: {status:8s} TIMESTAMP: {elapsed:6.1f}s")

            if max_seconds and (time.time() - t0) > max_seconds:
                break
    finally:
        cap.release()

print("predict_live() ready. Call predict_live(best_model, cfg, source=0) for a webcam, "
      "or source='rtsp://...' for an IP camera stream.")


## Ablation study — does Laya actually help?

Runs the same test set through four configurations and reports whether the added
complexity (object features, Laya verification) is earning its keep, rather than
assuming it does because it sounds impressive.

| System | What's on |
|---|---|
| Temporal only (Baseline A) | `AegisTemporalModel`, no YOLO, no Laya |
| + Objects (Baseline B) | + YOLO object-context features concatenated pre-fusion |
| + Laya (Baseline C) | Temporal model + Laya verification/fusion, no YOLO |
| Full system (Baseline D) | Temporal + YOLO + Laya |


In [ ]:
def run_ablation_study(model, test_loader, test_split, cfg: AegisConfig):
    rows = []

    configs = {
        "Temporal only":  dict(use_yolo=False, laya_enabled=False),
        "+ Objects":      dict(use_yolo=True,  laya_enabled=False),
        "+ Laya":         dict(use_yolo=False, laya_enabled=True),
        "Full system":    dict(use_yolo=True,  laya_enabled=True),
    }

    for name, overrides in configs.items():
        sub_cfg = AegisConfig(**{**cfg.__dict__, **overrides})

        t0 = time.time()
        _, macro_f1, (labels, preds), (anomaly_labels, anomaly_probs) = evaluate_model(model, test_loader, sub_cfg)
        latency_ms, fps = benchmark_inference_speed(model, sub_cfg, n_iters=10)

        try:
            auc = roc_auc_score(anomaly_labels, anomaly_probs)
        except ValueError:
            auc = None

        precision = classification_report(labels, preds, output_dict=True, zero_division=0)["weighted avg"]["precision"]
        recall = classification_report(labels, preds, output_dict=True, zero_division=0)["weighted avg"]["recall"]

        rows.append({
            "System": name, "F1": round(macro_f1, 3), "Recall": round(recall, 3),
            "Precision": round(precision, 3), "AUC": round(auc, 3) if auc else None,
            "FPS": round(fps, 1), "Latency_ms": round(latency_ms, 1),
        })
        print(f"{name}: F1={macro_f1:.3f}  FPS={fps:.1f}")

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(cfg.output_dir, "ablation_study.csv"), index=False)
    print("\n", df.to_string(index=False))
    print("\nNOTE: this harness re-scores the same trained temporal model with different "
          "auxiliary signals toggled off — YOLO/Laya are not separately trained here. If a "
          "row shows no F1 improvement over 'Temporal only', that's the honest result: don't "
          "keep the extra complexity in the deployed pipeline just because the plan called for it.")
    return df

# Run once training has produced `best_model` and `test_loader`:
# ablation_df = run_ablation_study(best_model, test_loader, test_split, cfg)


## Final summary

In [ ]:
def print_final_summary(cfg: AegisConfig, test_metrics: dict = None, fps: float = None):
    print("=" * 60)
    print("AEGIS TRAINING COMPLETE")
    print("=" * 60)
    print(f"Model:              AEGIS Video Anomaly Model ({cfg.backbone_name} + Transformer head)")
    print(f"Classes:            {', '.join(cfg.class_names)}")
    if test_metrics:
        print(f"Test macro F1:      {test_metrics.get('macro_f1')}")
        print(f"Test anomaly AUC:   {test_metrics.get('anomaly_roc_auc')}")
        print(f"Test anomaly PR-AUC:{test_metrics.get('anomaly_pr_auc')}")
    if fps:
        print(f"Inference FPS:      {fps}")
    ckpt_path = os.path.join(cfg.output_dir, "aegis_video_anomaly_model.pt")
    if os.path.exists(os.path.join(cfg.output_dir, "aegis_best.pt")):
        import shutil
        shutil.copy(os.path.join(cfg.output_dir, "aegis_best.pt"), ckpt_path)
    print(f"Checkpoint:         {ckpt_path}")
    print("=" * 60)

if total_videos > 0:
    print_final_summary(cfg, test_metrics if 'test_metrics' in dir() else None)
else:
    print("Nothing trained yet — populate the dataset paths in `cfg` and re-run from Cell Group 4.")


## Honest limitations of this stage

- The closed-set classifier only recognizes the exact categories it was trained on — a
  genuinely novel event type will fall back to the anomaly score at best, and may be
  missed entirely if it doesn't resemble the motion signature of any training class.
- Laya Vision's contribution is only real if `run_ablation_study` shows an actual F1/AUC
  gain over "Temporal only" on **your** trained model and **your** data. Don't keep it
  in the deployed pipeline on the strength of the architecture diagram alone.
- Frozen-backbone training is cheap but caps accuracy below what full fine-tuning could
  reach — the config exposes `freeze_backbone=False` to fine-tune the last block if
  Colab's GPU budget allows a slower training run.
- This is a detection layer, not a legal or safety system. False positives/negatives on
  violence and weapons detection have real consequences if this is ever connected to an
  actual alerting pipeline — treat any deployment beyond research/testing as requiring a
  much larger validation effort than one notebook can provide.
